# V1-S12 — Forecasting baselines (validation set)

This notebook **loads** the V1-S12 baseline artifacts produced upstream by the `scifield forecasting` CLI and renders the **validation** AUC + MAPE table, the per-split class balance, and the noise diagnostics. It performs **no network calls** and does **NOT** read the sealed 2021–2025 test set (there are no test rows in these artifacts — `materialize` ran with `allow_test=False`).

It does **not** recompute anything: the four baselines (naive / arima / mlp / no_graph), their metrics, and the materialized feature/label frame already exist on disk. We only read `forecasting_baselines.parquet`, `forecasting_features.parquet`, and both `.run.json` sidecars.

The **GNN comparison + Wilcoxon signed-rank** test against these baselines are **V1-S14**, not here. This session's job is to surface the validation floor and the class-balance / noise numbers for Samer to sanity-check before OSF submission.

## 1. Setup / load

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from scifield.repro import record_run  # noqa: E402

METRICS_PATH = DATA / "forecasting_baselines.parquet"
FEATURES_PATH = DATA / "forecasting_features.parquet"
METRICS_SIDECAR_PATH = Path(str(METRICS_PATH) + ".run.json")
FEATURES_SIDECAR_PATH = Path(str(FEATURES_PATH) + ".run.json")
DPI = 120


def _load_parquet(path: Path) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot render this section.")
        return None
    return pd.read_parquet(path)


def _load_json(path: Path) -> dict | None:
    """Defensive sidecar read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING sidecar: {path} — provenance fields unavailable.")
        return None
    return json.loads(path.read_text())


metrics = _load_parquet(METRICS_PATH)
features = _load_parquet(FEATURES_PATH)
metrics_sidecar = _load_json(METRICS_SIDECAR_PATH)
features_sidecar = _load_json(FEATURES_SIDECAR_PATH)

print(f"metrics parquet  : {METRICS_PATH}")
print(f"features parquet : {FEATURES_PATH}")
if metrics is not None:
    print(f"metrics rows     : {len(metrics)}  cols: {list(metrics.columns)}")
if features is not None:
    print(
        f"features rows    : {len(features):,}  split values: {sorted(features['split'].unique())}"
    )
    # Hard guard: this notebook must NEVER see test-origin rows.
    assert "test" not in set(
        features["split"].unique()
    ), "test rows present — sealed set must stay out"
    print("no-test-read assertion PASS — split values are a subset of {train, val}")

metrics parquet  : /Users/samersalman/Desktop/SciField/data/v1/forecasting_baselines.parquet
features parquet : /Users/samersalman/Desktop/SciField/data/v1/forecasting_features.parquet
metrics rows     : 4  cols: ['baseline', 'emergence_auc', 'share_mape', 'n_train', 'n_val', 'n_val_pos', 'fallback_frac']
features rows    : 3,264  split values: ['train', 'val']
no-test-read assertion PASS — split values are a subset of {train, val}


## 2. Validation performance table

One row per baseline on the **validation** set (origin years 2018–2020), sorted by `emergence_auc` descending. `emergence_auc` is ROC-AUC of the emergence score vs the binary `emergent` label; `share_mape` is mean absolute percentage error of the forward-share forecast (computed only where the target is positive). `fallback_frac` is NaN for every non-ARIMA baseline (only ARIMA can fall back to naive on degenerate series). The `n_train / n_val / n_val_pos` columns are identical across baselines — every predictor trains and evaluates on the same labeled slice (`volume_ok & label_complete`).

In [2]:
if metrics is None:
    print("metrics parquet missing — skipping validation performance table.")
else:
    cols = [
        "baseline",
        "emergence_auc",
        "share_mape",
        "fallback_frac",
        "n_train",
        "n_val",
        "n_val_pos",
    ]
    metrics_view = (
        metrics[cols].sort_values("emergence_auc", ascending=False).reset_index(drop=True)
    )
    print("Validation performance — one row per baseline (sorted by emergence_auc desc):\n")
    with pd.option_context("display.float_format", lambda v: f"{v:.6f}"):
        display(metrics_view)

    best = metrics_view.iloc[0]
    print(
        f"\nBest validation emergence-AUC: {best['baseline']} = {best['emergence_auc']:.4f}"
        f"  (this is the floor the V1-S13 GNN must beat by >5pp — Gate G4, tested in V1-S14)."
    )

Validation performance — one row per baseline (sorted by emergence_auc desc):



,baseline,emergence_auc,share_mape,fallback_frac,n_train,n_val,n_val_pos
0,no_graph,0.700955,0.364134,NaN,1298,208,17
1,arima,0.697259,0.289555,0.076923,1298,208,17
2,mlp,0.647367,0.325222,NaN,1298,208,17
3,naive,0.435479,0.281405,NaN,1298,208,17



Best validation emergence-AUC: no_graph = 0.7010  (this is the floor the V1-S13 GNN must beat by >5pp — Gate G4, tested in V1-S14).


In [3]:
# Bar chart of validation emergence-AUC per baseline -> saved to docs/figures.
F3_AUC_FIG = FIGURES_DIR / "f3_baselines_val_auc.png"

if metrics is None:
    print("metrics parquet missing — skipping AUC bar chart.")
else:
    plot_df = metrics.sort_values("emergence_auc", ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(
        plot_df["baseline"].to_numpy(),
        plot_df["emergence_auc"].to_numpy(),
        color="#4c72b0",
    )
    ax.axhline(0.5, color="red", ls="--", lw=1.0, label="chance (AUC=0.5)")
    ax.set_ylim(0, 1)
    ax.set_ylabel("validation emergence AUC")
    ax.set_xlabel("baseline")
    ax.set_title("F3 — V1-S12 baseline validation emergence AUC")
    for b, v in zip(bars, plot_df["emergence_auc"].to_numpy(), strict=False):
        ax.text(
            b.get_x() + b.get_width() / 2,
            v + 0.01,
            f"{v:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.legend(loc="upper right", framealpha=0.9)
    fig.tight_layout()
    fig.savefig(F3_AUC_FIG, dpi=DPI)
    plt.close(fig)
    print("wrote", F3_AUC_FIG, f"({F3_AUC_FIG.stat().st_size / 1024:.1f} KB)")

    # Record figure provenance, folding in the source artifacts' git/config hashes.
    fig_sidecar = record_run(
        artifact_path=F3_AUC_FIG,
        inputs={"metrics": METRICS_PATH, "features": FEATURES_PATH},
        config={
            "figure": "f3_baselines_val_auc",
            "session": "V1-S12",
            "split": "val",
            "baselines": plot_df["baseline"].tolist(),
            "emergence_auc": {
                r["baseline"]: float(r["emergence_auc"]) for _, r in plot_df.iterrows()
            },
            "dpi": DPI,
            "source_metrics_config_hash": (
                metrics_sidecar["config_hash"] if metrics_sidecar else None
            ),
            "source_metrics_git_sha": metrics_sidecar["git_sha"] if metrics_sidecar else None,
        },
    )
    print("recorded run sidecar:", fig_sidecar)

wrote /Users/samersalman/Desktop/SciField/docs/figures/f3_baselines_val_auc.png (29.2 KB)
recorded run sidecar: /Users/samersalman/Desktop/SciField/docs/figures/f3_baselines_val_auc.png.run.json


## 3. Class balance / per-split positive rates

The `per_split` block in the **features** sidecar carries each split's labeled-set size `n` and `positive_rate` (the fraction with `emergent == 1`). As a cross-check we recompute the positive rate **directly** from `forecasting_features.parquet` on the labeled slice (`volume_ok & label_complete`) per split, and display both side by side — they must agree. The sealed test split is absent (n=0).

In [4]:
if features_sidecar is None:
    print("features sidecar missing — cannot read per_split class balance.")
    per_split = {}
else:
    per_split = features_sidecar["config"]["per_split"]
    sidecar_balance = pd.DataFrame(
        [
            {
                "split": s,
                "sidecar_n": per_split[s]["n"],
                "sidecar_positive_rate": per_split[s]["positive_rate"],
            }
            for s in ("train", "val", "test")
            if s in per_split
        ]
    )
    print("Per-split class balance — from the features sidecar `per_split`:\n")
    with pd.option_context("display.float_format", lambda v: f"{v:.6f}"):
        display(sidecar_balance)

# Independent cross-check straight off the parquet labeled slice.
if features is None:
    print("features parquet missing — cannot cross-check positive rates.")
else:
    labeled = features[features["volume_ok"] & features["label_complete"]]
    grp = labeled.groupby("split")["emergent"].agg(parquet_n="size", parquet_positive_rate="mean")
    grp = grp.reindex([s for s in ("train", "val") if s in grp.index]).reset_index()
    print(
        "\nCross-check — recomputed from forecasting_features.parquet "
        "(volume_ok & label_complete):\n"
    )
    with pd.option_context("display.float_format", lambda v: f"{v:.6f}"):
        display(grp)

    # Assert the two sources agree on the train/val splits.
    if per_split:
        for _, row in grp.iterrows():
            s = row["split"]
            assert int(row["parquet_n"]) == int(per_split[s]["n"]), f"n mismatch on {s}"
            assert (
                abs(float(row["parquet_positive_rate"]) - float(per_split[s]["positive_rate"]))
                < 1e-9
            ), f"positive_rate mismatch on {s}"
        print("\ncross-check assertion PASS — sidecar per_split == parquet labeled-slice rates")

Per-split class balance — from the features sidecar `per_split`:



,split,sidecar_n,sidecar_positive_rate
0,train,1298,0.042373
1,val,208,0.081731
2,test,0,NaN



Cross-check — recomputed from forecasting_features.parquet (volume_ok & label_complete):



,split,parquet_n,parquet_positive_rate
0,train,1298,0.042373
1,val,208,0.081731



cross-check assertion PASS — sidecar per_split == parquet labeled-slice rates


In [5]:
# Threshold callout — train positive rate vs the plan's 5% floor.
if per_split and "train" in per_split:
    train_pr = float(per_split["train"]["positive_rate"])
    FLOOR = 0.05
    print("=" * 72)
    print("CALLOUT — train class balance vs plan floor")
    print("=" * 72)
    print(f"  train positive_rate = {train_pr:.4f}  (~0.042)")
    print(f"  plan floor          = {FLOOR:.2f}  (degenerate if <5% or >95%)")
    print(f"  below 5% floor?     -> {train_pr < FLOOR}")
    print("-" * 72)
    print(
        "  The train positive rate (~0.042) sits BELOW the plan's 5% floor at\n"
        "  GAMMA=1.5 / V_min=30. Per plan stop-condition, this is SAMER's threshold\n"
        "  call to retune (via conf/forecasting/v1.yaml) BEFORE OSF submission —\n"
        "  Claude does not silently change the thresholds."
    )
    print("=" * 72)
else:
    print("train per_split unavailable — skipping threshold-floor callout.")

CALLOUT — train class balance vs plan floor
  train positive_rate = 0.0424  (~0.042)
  plan floor          = 0.05  (degenerate if <5% or >95%)
  below 5% floor?     -> True
------------------------------------------------------------------------
  The train positive rate (~0.042) sits BELOW the plan's 5% floor at
  GAMMA=1.5 / V_min=30. Per plan stop-condition, this is SAMER's threshold
  call to retune (via conf/forecasting/v1.yaml) BEFORE OSF submission —
  Claude does not silently change the thresholds.


## 4. Noise diagnostics

Noise topic (`topic_id == -1`) totals from the **features** sidecar. The share denominator policy is `leaf_only`: noise papers are excluded from BOTH the numerator AND the denominator of every topic share, so each year's leaf shares sum to 1 and noise never dilutes the signal.

In [6]:
if features_sidecar is None:
    print("features sidecar missing — cannot read noise diagnostics.")
else:
    cfg = features_sidecar["config"]
    n_noise_total = cfg.get("n_noise_total")
    noise_frac = cfg.get("noise_frac")
    denominator = cfg.get("denominator")
    print("Noise diagnostics (from forecasting_features.parquet.run.json):")
    print(f"    n_noise_total : {n_noise_total:,}")
    print(f"    noise_frac    : {noise_frac:.4f}   (~0.240)")
    print(f"    denominator   : {denominator!r}")
    print()
    print(
        "Noise is excluded from BOTH the numerator AND the denominator of every "
        "topic share (leaf-only share),\nso per-year leaf shares sum to 1 and noise "
        "papers never enter the forecasting signal."
    )

Noise diagnostics (from forecasting_features.parquet.run.json):
    n_noise_total : 21,409
    noise_frac    : 0.2399   (~0.240)
    denominator   : 'leaf_only'

Noise is excluded from BOTH the numerator AND the denominator of every topic share (leaf-only share),
so per-year leaf shares sum to 1 and noise papers never enter the forecasting signal.


## 5. Pre-registration linkage & next step

Both V1-S12 artifacts carry the `preregistration` block in their `.run.json` sidecars. The OSF link is currently **`PENDING_OSF_SUBMISSION`** (see below): Samer uploads `docs/preregistrations/PR2_forecasting.md` to OSF, pastes the DOI into PR2 + `conf/forecasting/v1.yaml`, and a cheap CPU re-run of `scifield forecasting baselines` stamps the real DOI into the baseline sidecar.

**V1-S13 (the HGT/TGN GNN on the Brev A100) is blocked until that OSF DOI is committed.** The GNN-vs-baseline comparison and the Wilcoxon signed-rank test (Gate G4) are **V1-S14**; this notebook only establishes the validation floor those steps will be measured against.

In [7]:
def _prereg(sc: dict | None) -> dict:
    if not sc:
        return {}
    return sc.get("config", {}).get("preregistration", {})


feat_prereg = _prereg(features_sidecar)
metr_prereg = _prereg(metrics_sidecar)

print("Pre-registration linkage (from the artifact sidecars):")
print(f"    features sidecar osf_url : {feat_prereg.get('osf_url', '<absent>')}")
print(f"    baselines sidecar osf_url: {metr_prereg.get('osf_url', '<absent>')}")
print(
    "    pr_doc                   : "
    f"{metr_prereg.get('pr_doc', feat_prereg.get('pr_doc', '<absent>'))}"
)
print()
osf = metr_prereg.get("osf_url") or feat_prereg.get("osf_url")
if osf == "PENDING_OSF_SUBMISSION":
    print("OSF DOI is PENDING_OSF_SUBMISSION — V1-S13 (GNN) stays BLOCKED until the")
    print("real DOI is pasted in and a CPU re-run commits it into the baseline sidecar.")
else:
    print(f"OSF DOI committed: {osf} — V1-S13 may proceed.")

Pre-registration linkage (from the artifact sidecars):
    features sidecar osf_url : https://doi.org/10.17605/OSF.IO/XP94F
    baselines sidecar osf_url: https://doi.org/10.17605/OSF.IO/XP94F
    pr_doc                   : docs/preregistrations/PR2_forecasting.md

OSF DOI committed: https://doi.org/10.17605/OSF.IO/XP94F — V1-S13 may proceed.
